# SAM3 Faz 3 — Model + LoRA Kurulumu

Bu notebook Faz 3'ün tüm adımlarını içerir:
1. Paket kurulumu
2. HuggingFace login
3. SAM3 model yükleme testi
4. LoRA uygulama testi
5. Örnek görsel ile çıktı doğrulama

**ÖNEMLİ:** Çalıştırmadan önce üst menüden `Runtime > Change runtime type > T4 GPU` seç!

## Adım 0 — GPU Kontrolü

Önce GPU'nun aktif olduğunu doğrulayalım.

In [ ]:
import torch

if torch.cuda.is_available():
    print(f"GPU aktif: {torch.cuda.get_device_name(0)}")
    print(f"GPU bellek: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("UYARI: GPU bulunamadı! Runtime > Change runtime type > T4 GPU seç.")

## Adım 1 — Paket Kurulumu

Gerekli kütüphaneleri kuruyoruz. Bu birkaç dakika sürebilir.

In [ ]:
!pip install -q transformers peft accelerate

## Adım 2 — HuggingFace Login

SAM3 modeli gated (erişim kısıtlı) olduğu için token ile giriş yapmamız gerekiyor.

Token'ı güvenli şekilde Colab Secrets'a ekle:
1. Sol menüde 🔑 simgesine tıkla
2. `Add new secret` → Name: `HF_TOKEN`, Value: token'ını yapıştır
3. Notebook access'i aç (toggle)

In [ ]:
from huggingface_hub import login
from google.colab import userdata

# Token Colab Secrets'tan güvenli şekilde okunuyor
# Token koda yazılmıyor, dışarıdan görünmüyor
HF_TOKEN = userdata.get('HF_TOKEN')

login(token=HF_TOKEN)
print("HuggingFace girişi başarılı!")

## Adım 3 — SAM3 Model ve Processor Yükleme

Modeli HuggingFace'ten indirip yüklüyoruz. İlk seferde ~1-2 GB indirilecek, biraz bekle.

In [ ]:
from transformers import AutoModel, AutoProcessor

MODEL_NAME = "facebook/sam3"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Cihaz: {DEVICE}")
print(f"Model indiriliyor: {MODEL_NAME}")
print("Bu işlem birkaç dakika sürebilir...\n")

# Processor yükle
processor = AutoProcessor.from_pretrained(MODEL_NAME)
print("Processor yüklendi.")

# Model yükle
model = AutoModel.from_pretrained(MODEL_NAME)
model = model.to(DEVICE)
print(f"Model yüklendi ve {DEVICE} cihazına taşındı.")

## Adım 4 — Modelin Katman Adlarını İncele

LoRA uygularken hangi katmanları hedef alacağımızı bulmamız gerekiyor.
Bu hücre modelin içindeki katman adlarını listeler.

In [ ]:
print("Modeldeki katman adları (Linear olanlar):")
print("-" * 50)

import torch.nn as nn

gorulen_tipler = set()
for isim, katman in model.named_modules():
    if isinstance(katman, nn.Linear):
        # Sadece son kısmı göster (q_proj, v_proj gibi)
        son_kisim = isim.split(".")[-1]
        gorulen_tipler.add(son_kisim)

print("Bulunan Linear katman türleri:")
for tip in sorted(gorulen_tipler):
    print(f"  - {tip}")

## Adım 5 — LoRA Uygulama

Modele LoRA adaptörlerini ekliyoruz.
Bir önceki hücredeki çıktıya göre `target_modules` güncellenebilir.

In [ ]:
from peft import LoraConfig, get_peft_model

# LoRA ayarları (config.py ile aynı)
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["q_proj", "v_proj"],
)

# LoRA'yı modele uygula
model = get_peft_model(model, lora_config)

# Eğitilebilir parametre oranını göster
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
oran = 100 * trainable / total

print(f"{'='*50}")
print(f"Toplam parametreler      : {total:,}")
print(f"Eğitilebilir parametreler: {trainable:,}")
print(f"Eğitilebilir oran        : {oran:.2f}%")
print(f"{'='*50}")
print("\nBeklenen oran: ~%1-2. Bu aralıktaysa LoRA doğru çalışıyor!")

## Adım 6 — Örnek Görsel ile Test

Modelin gerçekten çalışıp çalışmadığını test etmek için örnek bir görsel kullanıyoruz.
Hata almadan çıktı geliyorsa Faz 3 tamamdır!

In [ ]:
from PIL import Image
import requests
from io import BytesIO

# Test için örnek bir görsel indir
gorsel_url = "https://images.unsplash.com/photo-1558618666-fcd25c85cd64?w=320"
response = requests.get(gorsel_url, headers={"User-Agent": "Mozilla/5.0"})
gorsel = Image.open(BytesIO(response.content)).convert("RGB")

print(f"Görsel boyutu: {gorsel.size}")
print("Görsel yüklendi.")

In [ ]:
# SAM3 bir video modelidir, inference için özel bir oturum gerektirir.
# Modelin vision encoder kısmını doğrudan test ediyoruz.
#
# Model yapısı:
# model.base_model.model.detector_model.vision_encoder → Sam3VisionModel
# model.base_model.model.detector_model.mask_decoder  → Sam3MaskDecoder
# model.base_model.model.tracker_model                → Sam3TrackerVideoModel

with torch.no_grad():
    pixel_values = inputs["pixel_values"].to(DEVICE)
    gorsel_ozellikleri = model.base_model.model.detector_model.vision_encoder(pixel_values)

print("Vision encoder çalışıyor!")
print(f"Çıktı tipi: {type(gorsel_ozellikleri)}")
print("\nFaz 3 TAMAMLANDI!")
print("- Model yüklendi")
print("- LoRA uygulandı")
print("- Vision encoder test edildi")

## Özet

Tüm hücreler hatasız çalıştıysa:
- SAM3 modeli yüklendi
- LoRA uygulandı
- Örnek görsel ile forward pass yapıldı

**Faz 3 tamamlandı. Faz 4'e geçilebilir!**